# Views, procedures, and transactions

As a database grows, so does the temptation to repeat the same query and the same multi-step
update. A **view** names a saved query, a **stored procedure** names a saved sequence of
statements, and a **transaction** groups statements so they all succeed or all fail. This
notebook builds each of those on top of the OpenFlights database.

## Learning objectives

By the end of this notebook you will be able to:

- create and use a view to hide a complex join;
- explain what a stored procedure is and how to get the same effect in SQLite;
- wrap writes in a transaction and roll them back on failure;
- describe the ACID properties in plain language;
- use a trigger to keep a derived value consistent.

## Concept

A **view** is a query stored under a name. It takes no space for the data itself and is evaluated
when selected, so it is a convenience and a security boundary (grant the view, not the tables).
The module's `schema.sql` already defines `v_route_details`, which joins routes to airlines and
airports.

A **stored procedure** is a named block of SQL kept in the database and invoked by name. PostgreSQL
and MySQL support them (and functions); SQLite does not. The practical equivalent is a Python
function that runs the statements inside one transaction, which also lets you validate inputs
before writing.

A **transaction** is a unit of work. `BEGIN` starts it, `COMMIT` makes it durable, and `ROLLBACK`
undoes everything since the beginning. Databases promise **ACID**: atomicity (all or nothing),
consistency (constraints hold), isolation (concurrent work does not interfere), and durability
(committed data survives a crash). Grouping related writes protects you from half-finished state.

A **trigger** is SQL that fires automatically on insert, update, or delete — useful for keeping a
derived column in step with its source.

## Worked example

### Connect and start from a clean scratch database

To keep the demonstration safe we copy the schema into an in-memory database, add a tiny amount of
data, and work there. The same statements run unchanged against the real file.

In [ ]:
import sqlite3
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from load import SCHEMA_PATH
from ds_practice import connect_sqlite, query
from ds_practice.paths import data_path
from load import build_database, DEFAULT_DB

db_path = data_path(DEFAULT_DB)
if not db_path.exists():
    db_path = build_database()
conn = connect_sqlite(db_path)

print("route-detail view exists:",
      bool(query(conn, "SELECT name FROM sqlite_master WHERE type='view'").shape[0]))
display(query(conn, "SELECT * FROM v_route_details LIMIT 3"))

### A view of our own

Rather than repeat a three-way join, we name it. `CREATE VIEW` can replace an earlier definition.

In [ ]:
conn.execute("DROP VIEW IF EXISTS v_country_routes")
conn.execute("""
    CREATE VIEW v_country_routes AS
    SELECT co.name AS country, COUNT(*) AS routes
    FROM routes r
    JOIN airports src ON src.airport_id = r.source_airport_id
    JOIN cities c     ON c.city_id = src.city_id
    JOIN countries co ON co.country_id = c.country_id
    GROUP BY co.country_id
""")
conn.commit()
display(query(conn, "SELECT * FROM v_country_routes ORDER BY routes DESC LIMIT 5"))

### A "procedure" as a Python function

SQLite has no `CREATE PROCEDURE`, so we express the same idea in Python: validate, then run the
writes inside one transaction. If anything raises, the `except` rolls everything back.

In [ ]:
def add_country(db, country_id: int, name: str) -> None:
    """Insert a country atomically. Roll back if the name already exists."""
    try:
        db.execute("BEGIN")
        db.execute("INSERT INTO countries (country_id, name) VALUES (?, ?)", (country_id, name))
        db.commit()
        print(f"inserted {name}")
    except sqlite3.IntegrityError as exc:
        db.rollback()
        print(f"rolled back: {exc}")

add_country(conn, 9001, "Testland")
add_country(conn, 9002, "Testland")   # UNIQUE(name) fails -> rollback
print("Testland rows:", query(conn, "SELECT COUNT(*) AS n FROM countries WHERE name='Testland'")["n"].iloc[0])

### Transaction and rollback

The classic demonstration: start a transaction, make a change, then roll it back and observe that
the table is unchanged. This is what protects a multi-step update from leaving half a result.

In [ ]:
before = query(conn, "SELECT COUNT(*) AS n FROM countries")["n"].iloc[0]

conn.execute("BEGIN")
conn.execute("INSERT INTO countries (country_id, name) VALUES (9003, 'Rollbackville')")
after_insert = query(conn, "SELECT COUNT(*) AS n FROM countries")["n"].iloc[0]
conn.rollback()
after_rollback = query(conn, "SELECT COUNT(*) AS n FROM countries")["n"].iloc[0]

print("before:", before, "| inside transaction:", after_insert, "| after rollback:", after_rollback)

### A trigger

A trigger keeps a derived column correct without the application remembering to update it. We add
an `insert_count` column to a small table and increment it automatically on every insert.

In [ ]:
conn.executescript("""
    DROP TABLE IF EXISTS audit_log;
    CREATE TABLE audit_log (
        log_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        event    TEXT NOT NULL,
        entries  INTEGER NOT NULL DEFAULT 0
    );
    DROP TRIGGER IF EXISTS trg_audit;
    CREATE TRIGGER trg_audit AFTER INSERT ON audit_log
    BEGIN
        UPDATE audit_log SET entries = entries + 1 WHERE log_id = NEW.log_id;
    END;
""")
for event in ("bootstrap", "load"):
    conn.execute("INSERT INTO audit_log (event) VALUES (?)", (event,))
conn.commit()
display(query(conn, "SELECT * FROM audit_log ORDER BY log_id"))

## Exercises

1. **Reusable view.** Create a view `v_routes_per_airline` returning `airline` and a route count,
   then select the top five from it without repeating the join.
2. **Atomic transfer.** Write a Python procedure that inserts a country and a city in one
   transaction, rolling back both if the city's foreign key is invalid. Verify the country is not
   left behind.
3. **Trigger guard.** Add a trigger that fires before an insert and rejects a negative airport
   altitude by raising an error with `RAISE(ABORT, ...)`.

## Limitations

SQLite's lack of stored procedures means business logic lives in the application, which is harder
to share across languages than a database procedure. Views over big joins can be slow because they
are recomputed on each access; a materialised view (not available by default in SQLite) trades
freshness for speed. Triggers are powerful but invisible: they can make writes have surprising
side effects, so document every one. Finally, ACID guarantees hold for a single database file but
not across a distributed system, where the trade-offs become much harder.